# BiLSTM Attention — Smart MCQ Solver


## Imports


In [1]:
import os
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.metrics import log_loss


## Set Seed


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## Wandb Setup


In [3]:
!pip install -q wandb


In [4]:
!pip install -q wandb

import wandb

WANDB_API_KEY = "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"
WANDB_ENTITY = "models-indian-institute-of-technology-madras481" 
WANDB_PROJECT = "23f3001092-t22026"    

wandb.login(key=WANDB_API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

## Load Data


In [5]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


Train shape: (2000, 8)
Test shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Data Cleaning


In [6]:
def clean_prompt(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = text.replace("based on the given context", "")
    text = text.replace("carefully", "")
    text = text.replace("among the listed options", "")
    text = text.replace("from the following choices", "")
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["group"] = train_df["prompt"].apply(clean_prompt)
test_df["group"]  = test_df["prompt"].apply(clean_prompt)


## Pairwise Data Preparation


In [7]:
def make_pairwise(df, train=True):
    rows = []
    for _, row in df.iterrows():
        for option in ["A", "B", "C", "D", "E"]:
            option_text = row[option]
            if pd.isna(option_text):
                continue
            rows.append({
                "id": row["id"],
                "group": row["group"],
                "question": row["prompt"],
                "option": option_text,
                "option_name": option,
                "text": "question: " + row["prompt"] + " [SEP] option: " + option_text,
                "label": int(option == row["answer"]) if train else -1,
            })
    return pd.DataFrame(rows)

pair_train = make_pairwise(train_df)
pair_test  = make_pairwise(test_df, train=False)

print("Train pairwise shape:", pair_train.shape)
pair_train.head()


Train pairwise shape: (10000, 7)


,id,group,question,option,option_name,text,label
0,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,A,question: Pick the best possible answer: What ...,0
1,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,B,question: Pick the best possible answer: What ...,1
2,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,C,question: Pick the best possible answer: What ...,0
3,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,D,question: Pick the best possible answer: What ...,0
4,1,pick the best possible answer what is martin h...,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,E,question: Pick the best possible answer: What ...,0


## Tokenization


In [8]:
max_features = 45000
max_len = 280

tokenizer = Tokenizer(num_words=max_features, oov_token="<UNK>")
tokenizer.fit_on_texts(pair_train["text"])
print("Vocabulary size:", len(tokenizer.word_index))


Vocabulary size: 3037


## Train Validation Split


In [9]:
X = tokenizer.texts_to_sequences(pair_train["text"])
X = pad_sequences(X, maxlen=max_len, padding="post", truncating="post")

y = pair_train["label"].values
groups = pair_train["group"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, valid_idx = next(gss.split(X, y, groups))

X_train = X[train_idx]
X_valid = X[valid_idx]
y_train = y[train_idx]
y_valid = y[valid_idx]

print("Train size:", X_train.shape[0])
print("Valid size:", X_valid.shape[0])


Train size: 8015
Valid size: 1985


## DataLoader


In [10]:
class MCQDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(MCQDataset(X_train, y_train), batch_size=96, shuffle=True, num_workers=0)
valid_loader = DataLoader(MCQDataset(X_valid, y_valid), batch_size=128, shuffle=False, num_workers=0)


## Model Definition


In [11]:
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=384, hidden_dim=192):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_norm = nn.LayerNorm(embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3,
        )
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout1 = nn.Dropout(0.4)
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self.dropout2 = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        x = self.embedding_norm(x)
        out, _ = self.lstm(x)
        attn_scores = self.attention(out).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context = torch.sum(attn_weights.unsqueeze(-1) * out, dim=1)
        context = self.layer_norm(context)
        context = self.dropout1(context)
        return self.fc(context).squeeze(1)


## Model Setup


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = BiLSTMAttention(max_features).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-5)


Using device: cuda


## Helper Functions


In [13]:
def get_map3_score(model, loader, device):
    model.eval()
    valid_scores = []
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            probs = torch.sigmoid(outputs)
            valid_scores.extend(probs.cpu().numpy())

    valid_scores = np.array(valid_scores)
    valid_df_temp = pair_train.iloc[valid_idx].copy().reset_index(drop=True)
    valid_df_temp["score"] = valid_scores

    predictions, true_answers = [], []
    for _, group in valid_df_temp.groupby("id"):
        group = group.sort_values("score", ascending=False)
        predictions.append(group["option_name"].tolist()[:3])
        true_answers.append(group.loc[group["label"] == 1, "option_name"].values[0])

    score = 0.0
    for true, pred in zip(true_answers, predictions):
        if true == pred[0]:
            score += 1.0
        elif true == pred[1]:
            score += 0.5
        elif true == pred[2]:
            score += 1.0 / 3.0
    return score / len(true_answers)

def map3(y_true, y_pred):
    score = 0.0
    for true, pred in zip(y_true, y_pred):
        if true == pred[0]:
            score += 1.0
        elif true == pred[1]:
            score += 0.5
        elif true == pred[2]:
            score += 1.0 / 3.0
    return score / len(y_true)


## Training with Wandb Logging


In [14]:
epochs = 20
best_map3 = 0.0
patience = 5
patience_counter = 0

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="bilstm-attention",
    config={
        "model": "BiLSTM-Attention",
        "max_features": max_features,
        "max_len": max_len,
        "embedding_dim": 384,
        "hidden_dim": 192,
        "batch_size": 96,
        "epochs": epochs,
        "learning_rate": 3e-3,
        "weight_decay": 1e-4,
        "patience": patience,
        "seed": SEED,
    },
)


wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260714_191517-naeraknd
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bilstm-attention
wandb: ⭐️ View project at https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026
wandb: 🚀 View run at https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026/runs/naeraknd


In [15]:
for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    avg_train_loss = train_loss / len(train_loader)
    val_map3 = get_map3_score(model, valid_loader, device)

    print(f"Epoch {epoch+1}/{epochs}  Loss: {avg_train_loss:.4f}  Val MAP@3: {val_map3:.5f}  LR: {optimizer.param_groups[0]['lr']:.2e}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_map3": val_map3,
        "learning_rate": optimizer.param_groups[0]["lr"],
    })

    if val_map3 > best_map3:
        best_map3 = val_map3
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best MAP@3: {best_map3:.5f}")
            model.load_state_dict(torch.load("best_model.pt"))
            break

print(f"Training complete. Best MAP@3: {best_map3:.5f}")
wandb.log({"best_map3": best_map3})
wandb.finish()


Epoch 1/20  Loss: 0.5396  Val MAP@3: 0.72376  LR: 2.98e-03
Epoch 2/20  Loss: 0.3674  Val MAP@3: 0.91898  LR: 2.93e-03
Epoch 3/20  Loss: 0.1569  Val MAP@3: 0.95802  LR: 2.84e-03
Epoch 4/20  Loss: 0.0800  Val MAP@3: 0.97691  LR: 2.71e-03
Epoch 5/20  Loss: 0.0513  Val MAP@3: 0.98195  LR: 2.56e-03
Epoch 6/20  Loss: 0.0443  Val MAP@3: 0.98573  LR: 2.38e-03
Epoch 7/20  Loss: 0.0352  Val MAP@3: 0.98741  LR: 2.18e-03
Epoch 8/20  Loss: 0.0293  Val MAP@3: 0.98237  LR: 1.97e-03
Epoch 9/20  Loss: 0.0333  Val MAP@3: 0.98866  LR: 1.74e-03
Epoch 10/20  Loss: 0.0245  Val MAP@3: 0.98363  LR: 1.51e-03
Epoch 11/20  Loss: 0.0184  Val MAP@3: 0.98363  LR: 1.27e-03
Epoch 12/20  Loss: 0.0150  Val MAP@3: 0.98615  LR: 1.04e-03
Epoch 13/20  Loss: 0.0136  Val MAP@3: 0.98741  LR: 8.26e-04


wandb: updating run metadata


Epoch 14/20  Loss: 0.0109  Val MAP@3: 0.98866  LR: 6.26e-04
Early stopping at epoch 14. Best MAP@3: 0.98866
Training complete. Best MAP@3: 0.98866


wandb: uploading history steps 13-14, summary, console lines 13-15
wandb: 
wandb: Run history:
wandb:     best_map3 ▁
wandb:         epoch ▁▂▂▃▃▄▄▅▅▆▆▇▇█
wandb: learning_rate ███▇▇▆▆▅▄▄▃▂▂▁
wandb:    train_loss █▆▃▂▂▁▁▁▁▁▁▁▁▁
wandb:      val_map3 ▁▆▇███████████
wandb: 
wandb: Run summary:
wandb:     best_map3 0.98866
wandb:         epoch 14
wandb: learning_rate 0.00063
wandb:    train_loss 0.01089
wandb:      val_map3 0.98866
wandb: 
wandb: 🚀 View run bilstm-attention at: https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026/runs/naeraknd
wandb: ⭐️ View project at: https://wandb.ai/models-indian-institute-of-technology-madras481/23f3001092-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260714_191517-naeraknd/logs


## Validation Evaluation


In [16]:
model.eval()
valid_scores = []

with torch.no_grad():
    for X_batch, _ in valid_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = torch.sigmoid(outputs)
        valid_scores.extend(probs.cpu().numpy())

valid_scores = np.array(valid_scores)
print(f"Best model validation MAP@3: {best_map3:.5f}")
print(f"Valid scores sample: {valid_scores[:10]}")


Best model validation MAP@3: 0.98866
Valid scores sample: [1.2910237e-04 9.9976677e-01 1.5879102e-04 1.6441387e-04 1.9431039e-04
 9.9971193e-01 2.7583766e-04 2.7336282e-04 2.6369875e-04 2.6483889e-04]


In [17]:
valid_df = pair_train.iloc[valid_idx].copy().reset_index(drop=True)
valid_df["score"] = valid_scores
print(valid_df[["id", "option_name", "label", "score"]].head(10))
print("Valid df shape:", valid_df.shape)


   id option_name  label     score
0   4           A      0  0.000129
1   4           B      1  0.999767
2   4           C      0  0.000159
3   4           D      0  0.000164
4   4           E      0  0.000194
5   5           A      1  0.999712
6   5           B      0  0.000276
7   5           C      0  0.000273
8   5           D      0  0.000264
9   5           E      0  0.000265
Valid df shape: (1985, 8)


In [18]:
predictions = []
true_answers = []

for _, group in valid_df.groupby("id"):
    group = group.sort_values("score", ascending=False)
    predictions.append(group["option_name"].tolist()[:3])
    true_answers.append(group.loc[group["label"] == 1, "option_name"].values[0])

map_score = map3(true_answers, predictions)
top1_acc = np.mean(np.array([p[0] for p in predictions]) == np.array(true_answers))

print("Validation MAP@3:", round(map_score, 5))
print("Validation Accuracy:", round(top1_acc, 5))


Validation MAP@3: 0.98866
Validation Accuracy: 0.98489


## Test Prediction


In [19]:
test_sequences = tokenizer.texts_to_sequences(pair_test["text"])
test_sequences = pad_sequences(test_sequences, maxlen=max_len, padding="post", truncating="post")

test_loader = DataLoader(torch.LongTensor(test_sequences), batch_size=256, shuffle=False)


In [20]:
model.eval()
test_scores = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = torch.sigmoid(outputs)
        test_scores.extend(probs.cpu().numpy())

pair_test["score"] = test_scores

predictions = []
ids = []
for qid, group in pair_test.groupby("id"):
    group = group.sort_values("score", ascending=False)
    ids.append(qid)
    predictions.append(" ".join(group["option_name"].tolist()[:3]))


## Submission


In [21]:
submission = pd.DataFrame({"ID": ids, "Prediction": predictions})
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("submission.csv saved")


   ID Prediction
0   1      A E B
1   2      B A C
2   3      B A D
3   4      E D C
4   5      C A E
submission.csv saved
